In [2]:
using QuantumToolbox
using LinearAlgebra

In [3]:
X = [0.0 1.0; 1.0 0.0]
Y = [0.0 1.0*-im; 1.0*im 0.0]
Z = [1.0 0.0; 0.0 -1.0]
su2 = [1im*X, 1im*Y, 1im*Z]
coeffs = [1.,2.,3.]
exponent = sum(coeffs.*su2)
U = exp(exponent)

if U*U' ≈ I
    return true
else 
    return false
end

true

In [4]:
coeffs2 = [1.0, 2.0+ 1im, 3.0]
exponent = sum(coeffs2.*su2) 
U = exp(exponent)
if U*U' ≈ I
    return true
else 
    return false
end

false

# DRAG vs GAUSSIAN vs GRAPE

In [5]:
using Piccolo
using CairoMakie


SYSTEM: caught exception of type :MethodError while trying to print a failed Task notice; giving up

SYSTEM: caught exception of type :MethodError while trying to print a failed Task notice; giving up

SYSTEM: caught exception of type :MethodError while trying to print a failed Task notice; giving up

SYSTEM: caught exception of type :MethodError while trying to print a failed Task notice; giving up

SYSTEM: caught exception of type :MethodError while trying to print a failed Task notice; giving up


In [6]:
const Levels = 3
const Delta = 0.2
const Drive_bound = 0.5

sys = TransmonSystem(drive_bounds = [Drive_bound, Drive_bound], levels = Levels, δ = Delta)

U_goal = EmbeddedOperator(:X, sys) #X gate acting on |0>,|1> computational subspace

#intgrate a fucntion over [0,tg] with trapezoid rule used to claobrate the pulse area 
function trapezoid(f, tg, n=2000)
    ts = range(0, tg; length = n)
    ys = f.(ts)
    dt = tg/(n-1)
    return dt * (sum(ys)- 0.5*(ys[1] + ys[end]))
end

function gaussian_pulse(tg; sigma_frac = 0.25) # are aclaibrates the mplitude to pi(for X Gate)
    sigma = sigma_frac*tg
    shape(t) = exp(-(t-tg/2)^2/(2*sigma^2))
    area_unit = trapezoid(shape, tg)
    A = pi/area_unit
    return GaussianPulse([A,0.0],sigma,tg;center = tg/2)
end

function drag_pulse(tg,Δ;sigma_frac = 0.25, beta = 0.5)
    sigma = sigma_frac*tg
    shape(t) = exp(-(t-tg/2)^2/(2*sigma^2))
    dshape(t) = -(t-tg/2)/sigma^2*shape(t)
    area_unit = trapezoid(shape,tg)
    A = pi/area_unit

    f = t -> begin
        if t< 0 || t>tg
            return [0.0,0.0]
        end
        Ωx = A *shape(t)
        Ωy = -(A/Δ)*shape(t)
        return [Ωx,Ωy]
    end
    return FunctionPulse(f,tg,2)
end

function gate_error(pulse,tg)
    qtraj = UnitaryTrajectory(sys,pulse, U_goal)
    out = rollout(qtraj)
    return 1-fidelity(out)
end

function optimized_error(tg, Δ; N =60,max_iter =150)
    drag = drag_pulse(tg, Δ)
    times = collect(range(0,tg,length =N))
    ctrl = hcat([drag(t) for t in times]...)
    
    qtraj = qtraj = UnitaryTrajectory(sys,ZeroOrderPulse(ctrl, times),U_goal)
    opts = PiccoloOptions(display = :silent)
    qcp = SmoothPulseProblem(qtraj,N; Q =100.0, R = 1e-3, piccolo_options = opts)
    solve!(qcp; max_iter =max_iter, print_level = 1, eval_hessian = false)
    println("Intial", fidelity(qtraj))
    println("OpIntial", fidelity(qcp))
    return 1 - fidelity(qcp)
end



optimized_error (generic function with 1 method)

In [7]:
tg = 6.0

for scale in [0.25,0.5,1.0,2.0,4.0]
    sigma = 0.5*tg
    shape(t) = exp(-(t-tg/2)^2 / (2*sigma^2))
    A = scale *pi / trapezoid(shape,tg)
    f = t -> (0<=t<=tg) ? [A*shape(t),0.0] : [0.0,0.0]
    p = FunctionPulse(f,tg,2)
    println("scale = $scale -> error = ", gate_error(p,tg))
end

UndefVarError: UndefVarError: `fidelity` not defined in `Main`
Hint: It looks like two or more modules export different bindings with this name, resulting in ambiguity. Try explicitly importing it from a particular module, or qualifying the name with the module it should come from.
Hint: a global variable of this name also exists in QuantumToolbox.
Hint: a global variable of this name also exists in Piccolo.Quantum.Rollouts.
    - Also exported by Piccolo.

In [8]:
function to_zoh(pulse_fn, tg, N)
    times = collect(range(0,tg,length=N))
    ctrl = hcat([pulse_fn(t) for t in times]...)
    return ZeroOrderPulse(ctrl,times)
end
function gate_error_res(pulse_fn,tg,N)
    qtraj = UnitaryTrajectory(sys, to_zoh(pulse_fn,tg,N), U_goal)
    return 1 - fidelity(rollout(qtraj))
end
tg = 6.0
sigma = 0.5*tg
shape(t) = exp(-(t-tg/2)^2 / (2*sigma^2))
A = pi / trapezoid(shape,tg)
f = t -> (0<=t<=tg) ? [A*shape(t),0.0] : [0.0,0.0]

for N in [20,50,100,200,400,800]
    println("N=$N -> 1-F = ", gate_error_res(f,tg,N))
end

UndefVarError: UndefVarError: `fidelity` not defined in `Main`
Hint: It looks like two or more modules export different bindings with this name, resulting in ambiguity. Try explicitly importing it from a particular module, or qualifying the name with the module it should come from.
Hint: a global variable of this name also exists in QuantumToolbox.
Hint: a global variable of this name also exists in Piccolo.Quantum.Rollouts.
    - Also exported by Piccolo.

In [9]:

scales = range(0.05,5.0, length = 60)
errs = Float64[]

for scale in scales
    A_s = scale * A
    f = t -> (0<=t<=tg) ? [A_s*shape(t),0.0] : [0.0,0.0]
    push!(errs, gate_error_res(f,tg,400))
end

fig = Figure(); ax = Axis(fig[1,1],xlabel="scale", ylabel="1-F")
lines!(ax,scales,errs); display(fig)

UndefVarError: UndefVarError: `fidelity` not defined in `Main`
Hint: It looks like two or more modules export different bindings with this name, resulting in ambiguity. Try explicitly importing it from a particular module, or qualifying the name with the module it should come from.
Hint: a global variable of this name also exists in QuantumToolbox.
Hint: a global variable of this name also exists in Piccolo.Quantum.Rollouts.
    - Also exported by Piccolo.

In [11]:
gate_times = collect(range(3.0,9.0,length=13))
Δ = -Delta

err_gauss = Float64[]
err_drag = Float64[]
err_opt = Float64[]

for tg in gate_times
    push!(err_gauss, gate_error(gaussian_pulse(tg),tg))
    push!(err_drag, gate_error(drag_pulse(tg,Δ),tg))
    push!(err_opt, optimized_error(tg,Δ))
end

fig = Figure(size = (700,500))
ax = Axis(fig[1, 1], 
    xlabel = "Gate time, t_g", 
    ylabel = "gate error, 1-F", 
    yscale = log10, 
    title = "Gate error vs duration: Gaussian vs DRAG vs Optimized")

lines!(ax, gate_times, err_gauss, label = "Gaussian(no correction)", color =:blue)
scatter!(ax, gate_times, err_gauss, color=:blue)

lines!(ax,gate_times, err_drag, label = "Gaussian + DRAG", color =:red, linestyle = :dash)
scatter!(ax,gate_times, err_drag, color=:red)

lines!(ax,gate_times, err_opt, label = "Numerically optimized", color =:purple, linestyle = :dot)
scatter!(ax,gate_times, err_opt, color=:purple)

axislegend(ax,position = :rt, frame_visible = false)
display(fig)
#save("project1_gate_error_vs_duration",fig)
println("Saved")

UndefVarError: UndefVarError: `fidelity` not defined in `Main`
Hint: It looks like two or more modules export different bindings with this name, resulting in ambiguity. Try explicitly importing it from a particular module, or qualifying the name with the module it should come from.
Hint: a global variable of this name also exists in QuantumToolbox.
Hint: a global variable of this name also exists in Piccolo.Quantum.Rollouts.
    - Also exported by Piccolo.

In [12]:
gate_times = collect(range(3.0,9.0,length=13))
for tg in gate_times
    drag = drag_pulse(tg, Delta)
    println(drag(tg/4))
    println(drag(tg/2))
    println(drag(3tg/4))
end


[1.0618805211695745, -5.309402605847873]
[1.7507450021944153, -8.753725010972076]
[1.0618805211695745, -5.309402605847873]
[0.9101833038596352, -4.550916519298176]
[1.5006385733094987, -7.503192866547493]
[0.9101833038596352, -4.550916519298176]
[0.7964103908771809, -3.9820519543859043]
[1.3130587516458114, -6.565293758229057]
[0.7964103908771809, -3.9820519543859043]
[0.7079203474463829, -3.5396017372319144]
[1.1671633347962767, -5.835816673981383]
[0.7079203474463829, -3.5396017372319144]
[0.6371283127017445, -3.1856415635087223]
[1.050447001316649, -5.252235006583244]
[0.6371283127017445, -3.1856415635087223]
[0.579207557001586, -2.89603778500793]
[0.9549518193787718, -4.774759096893859]
[0.579207557001586, -2.89603778500793]
[0.5309402605847873, -2.6547013029239364]
[0.8753725010972077, -4.376862505486038]
[0.5309402605847873, -2.6547013029239364]
[0.49009870207826517, -2.4504935103913255]
[0.8080361548589609, -4.040180774294804]
[0.49009870207826517, -2.4504935103913255]
[0.455091

# DRAG breakdown vs PWC

In [13]:
leak_indices = get_iso_vec_leakage_indices(U_goal)

8-element Vector{Int64}:
  3
  6
  9
 12
 13
 14
 16
 17

In [14]:
drag_error(tg) = begin
    qtraj = UnitaryTrajectory(sys,drag_pulse(tg,Δ),U_goal)
    1 - fidelity(rollout(qtraj))
end
function pwc_from_drag_error(tg,N =40, max_iter = 200, leakage_cost =20.0, leakage_constraint_value = 1e-3) #20
    drag = drag_pulse(tg, Δ)
    times = collect(range(0,tg,length =N))
    ctrl = hcat([drag(t) for t in times]...)
    
    qtraj = qtraj = UnitaryTrajectory(sys,ZeroOrderPulse(ctrl, times),U_goal)
    opts = PiccoloOptions(
        leakage_cost = leakage_cost,
        leakage_constraint = true,
        leakage_constraint_value = leakage_constraint_value,
        display = :silent,
    )
    qcp = SmoothPulseProblem(qtraj,N; Q =100.0, R = 1e-3, piccolo_options = opts)
    solve!(qcp; max_iter =max_iter, print_level = 1, eval_hessian = false)
    return 1 - fidelity(qcp),qcp
end

pwc_from_drag_error (generic function with 5 methods)

In [15]:
gate_times_short = collect(range(1.7,4.5,length=10))

err_drag_short = Float64[]
err_pwc = Float64[]
last_qcp = nothing

for tg in gate_times_short
    ed = drag_error(tg)
    ep, qcp = pwc_from_drag_error(tg)
    global last_qcp = qcp
    push!(err_drag_short,ed)
    push!(err_pwc,ep)
end

UndefVarError: UndefVarError: `fidelity` not defined in `Main`
Hint: It looks like two or more modules export different bindings with this name, resulting in ambiguity. Try explicitly importing it from a particular module, or qualifying the name with the module it should come from.
Hint: a global variable of this name also exists in QuantumToolbox.
Hint: a global variable of this name also exists in Piccolo.Quantum.Rollouts.
    - Also exported by Piccolo.

In [16]:
fig2 = Figure(size = (700,500))
ax2 = Axis(fig2[1,1], xlabel = "Gate time",ylabel = "Gate_error",yscale = log10, title = "DRAG breakdown vs PWC coreection at short gate times")

lines!(ax2,gate_times_short, err_drag_short,label = "DRAG Only", color = :blue, linewidth = 4.0)
scatter!(ax2, gate_times_short, err_drag_short, color =:blue)


lines!(ax2,gate_times_short, err_pwc,label = "PWC corrextion on top of DRAG", color = :red)
scatter!(ax2, gate_times_short, err_pwc, color =:red)

axislegend(ax2, position = :lt)
display(fig2)
#save("project_drag_vs_pwc_short", fig2)

DimensionMismatch: DimensionMismatch: arrays could not be broadcast to a common size: a has axes Base.OneTo(10) and b has axes Base.OneTo(0)

In [17]:
#=
shortest_tg = gate_times,shorts[1]

_, qcp_short = pwc_from_drag(shortest_tg)
plot_unitary_populations(get_trajexctory(qcp_short))

save("project2, snallest pwc shortest_gate", current_figure())
=#

In [18]:
import Pkg
#Pkg.gc()
Pkg.develop(path="/Users/jettajb1/Work/DiffEQ/HarmonicSolvers/")

┌ Warning: could not download https://pkg.julialang.org/registries
│   exception = Downloads.RequestError("https://pkg.julialang.org/registries", 60, "SSL certificate problem: unable to get local issuer certificate", Downloads.Response("https", "https://pkg.julialang.org/registries", 0, "", Pair{String, String}[]))
└ @ Pkg.Registry /Users/jettajb1/.julia/juliaup/julia-1.12.6+0.aarch64.apple.darwin14/Julia-1.12.app/Contents/Resources/julia/share/julia/stdlib/v1.12/Pkg/src/Registry/Registry.jl:83
    Updating registry at `~/.julia/registries/General`
┌ Info: The General registry is installed via git. Consider reinstalling it via
│ the newer faster direct from tarball format by running:
│   pkg> registry rm General; registry add General
│ 
└ @ Pkg.Registry /Users/jettajb1/.julia/juliaup/julia-1.12.6+0.aarch64.apple.darwin14/Julia-1.12.app/Contents/Resources/julia/share/julia/stdlib/v1.12/Pkg/src/Registry/Registry.jl:505
    Updating git-repo `https://github.com/JuliaRegistries/General.git

# Single vs Dual Axis Control

In [19]:
const Levels = 3
const Delta = 0.2
const Drive_bound = 0.5
const Δ_freq =-1.0*2*pi*0.4 #-0.8pi radians

sys_2d = TransmonSystem(drive_bounds = [Drive_bound, Drive_bound], levels = Levels, δ = Delta)
U_goal = EmbeddedOperator(:Y, sys) #X gate acting on |0>,|1> computational subspace


for tg in [3.0,5.0,7.0,9.0]
    sigma = 0.5*tg
    shape(t) = exp(-(t-tg/2)^2/(2*sigma^2))
    A = pi/trapezoid(shape,tg)
    println("tgt=$tg peak amplitude needed = $A (bound = $Drive_bound)")
end

function pwc_comp_solver(tg; N = 40, max_iter = 200, use_y_axis = true)

    sys = use_y_axis ? TransmonSystem(drive_bounds = [Drive_bound, Drive_bound], levels = Levels, δ = Delta)  : TransmonSystem(drive_bounds = [Drive_bound, 0.0], levels = Levels, δ = Delta)
    drag = drag_pulse(tg, Δ_freq)
    times = collect(range(0,tg,length =N))
    ctrl=hcat([drag(t) for t in times]...)

    if !use_y_axis
        ctrl[2, :] .= 0.0
    end
    warm_pulse = ZeroOrderPulse(ctrl, times)
    qtraj = UnitaryTrajectory(sys, warm_pulse,U_goal)

    opts = PiccoloOptions(
        leakage_cost = 50.0,
        leakage_constraint = true,
        leakage_constraint_value = 1e-5,
        display = :silent,
    )

    qcp = SmoothPulseProblem(qtraj, N; Q = 100.0, R = 1e-3, piccolo_options = opts)
    solve!(qcp; max_iter = max_iter, print_level = 0, eval_hessian = false)
    return 1-fidelity(qcp)
end


tgt=3.0 peak amplitude needed = 1.2238987376268198 (bound = 0.5)
tgt=5.0 peak amplitude needed = 0.734339242576092 (bound = 0.5)
tgt=7.0 peak amplitude needed = 0.5245280304114942 (bound = 0.5)
tgt=9.0 peak amplitude needed = 0.40796624587560665 (bound = 0.5)


pwc_comp_solver (generic function with 1 method)

In [20]:
gate_times = collect(range(1.7,4.7,length =8))

err_x_only = Float64[]
err_xy = Float64[]


for tg in gate_times
    ep_x = pwc_comp_solver(tg,use_y_axis = false)
    ep_xy = pwc_comp_solver(tg,use_y_axis = true)

    push!(err_x_only,ep_x)
    push!(err_xy,ep_xy)
end



┌ Warning: Trajectory has timestep variable :Δt but no bounds on it.
│ Adding default lower bound of 0 to prevent negative timesteps.
│ 
│ Recommended: Add explicit bounds when creating the trajectory:
│   NamedTrajectory(...; Δt_bounds=(min, max))
│ Example:
│   NamedTrajectory(qtraj, N; Δt_bounds=(1e-3, 0.5))
│ 
│ Or use timesteps_all_equal=true in problem options to fix timesteps.
└ @ DirectTrajOpt.Problems /Users/jettajb1/.julia/packages/DirectTrajOpt/VOKy2/src/problems.jl:66



******************************************************************************
This program contains Ipopt, a library for large-scale nonlinear optimization.
 Ipopt is released as open source code under the Eclipse Public License (EPL).
         For more information visit https://github.com/coin-or/Ipopt
******************************************************************************



UndefVarError: UndefVarError: `fidelity` not defined in `Main`
Hint: It looks like two or more modules export different bindings with this name, resulting in ambiguity. Try explicitly importing it from a particular module, or qualifying the name with the module it should come from.
Hint: a global variable of this name also exists in QuantumToolbox.
Hint: a global variable of this name also exists in Piccolo.Quantum.Rollouts.
    - Also exported by Piccolo.

In [21]:
fig3 = Figure(size =(750,520))

ax = Axis(fig3[1,1], xlabel = "Gate Time", ylabel = "Gate Error", yscale = log10, title = "Control Field Axis Comparsion")
lines!(ax, gate_times, err_x_only, label = "X-Only Control", color = :darkred, linewidth =2.5)
scatter!(ax, gate_times, err_x_only, color = :darkred)

lines!(ax, gate_times, err_xy, label = "XY Control", color = :darkblue, linewidth =2.5)
scatter!(ax, gate_times, err_xy, color = :darkblue)

axislegend(ax, position = :lt)
save("project3.png", fig3)
fig3


DimensionMismatch: DimensionMismatch: arrays could not be broadcast to a common size: a has axes Base.OneTo(8) and b has axes Base.OneTo(0)